# Bird Tracking with YOLOv5 and Kalman Filter
This notebook demonstrates real-time object tracking using YOLOv5 for object detection and a Kalman Filter for temporal tracking.
We apply this pipeline to detect and track a bird in video footage, even when frames are missed or the detection confidence drops.

**Key Components:**
- YOLOv5m via PyTorch Hub for object detection
- Kalman Filter using FilterPy for motion estimation
- OpenCV for video frame processing and output


In [1]:
# Libraries

In [48]:
import cv2
import os
import numpy as np

import torch

import os
import cv2
from PIL import Image
from torchvision import transforms as T

In [3]:
# Local Path

In [4]:
# local_path = ""

In [5]:
# Split video into Frames

In [6]:
# def split_video_to_frames(video_path, output_dir):
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)

#     cap = cv2.VideoCapture(video_path)
#     frame_count = 0

#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         frame_path = os.path.join(output_dir, f"frame_{frame_count:04d}.png")
#         cv2.imwrite(frame_path, frame)
#         frame_count += 1

#     cap.release()
#     print(f"Extracted {frame_count} frames from {video_path}")

# video_paths =  ["/219828_small.mp4"]


# for video_path in video_paths:
#     output_dir = os.path.splitext(video_path)[0] + "_frames"
#     split_video_to_frames(video_path, output_dir)

In [ ]:
def split_video_to_frames(video_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_path = os.path.join(output_dir, f"frame_{frame_count:04d}.png")
        cv2.imwrite(frame_path, frame)  # Save frame in original resolution
        frame_count += 1

    cap.release()
    print(f"Extracted {frame_count} frames from {video_path}")

video_paths = ["/219828_small.mp4"]

for video_path in video_paths:
    output_dir = os.path.splitext(video_path)[0] + "_frames"
    split_video_to_frames(video_path, output_dir)


In [7]:
# Preprocess Frames

In [8]:
# def preprocess_frames(input_dir, output_file):
#     frame_files = [f for f in os.listdir(input_dir) if f.endswith('.png')]
#     frame_files.sort()

#     processed_frames = []
#     for frame_file in frame_files:
#         frame_path = os.path.join(input_dir, frame_file)
#         frame = cv2.imread(frame_path)

#         # Resize frame
#         resized_frame = cv2.resize(frame, (224, 224))

#         # Convert color from BGR to RGB
#         rgb_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2RGB)

#         # Normalize the frame
#         normalized_frame = rgb_frame / 255.0

#         processed_frames.append(normalized_frame)

#     # Save processed frames to a .npy file
#     np.save(output_file, np.array(processed_frames))
#     print(f"Saved processed frames to {output_file}")

# for video_path in video_paths:
#     input_dir = os.path.splitext(video_path)[0] + "_frames"
#     output_file = os.path.splitext(video_path)[0] + "_frames.npy"
#     preprocess_frames(input_dir, output_file)

In [50]:
def preprocess_frames(input_dir, output_file):
    frame_files = [f for f in os.listdir(input_dir) if f.endswith('.png')]
    frame_files.sort()

    processed_frames = []
    for frame_file in frame_files:
        frame_path = os.path.join(input_dir, frame_file)
        frame = cv2.imread(frame_path)

        # Resize frame to higher resolution
        resized_frame = cv2.resize(frame, (1024, 1024))  # Use larger resolution

        # Append the resized frame directly
        processed_frames.append(resized_frame)

    # Save processed frames to a .npy file
    np.save(output_file, np.array(processed_frames, dtype=np.uint8))
    print(f"Saved processed frames to {output_file}")

for video_path in video_paths:
    input_dir = os.path.splitext(video_path)[0] + "_frames"
    output_file = os.path.splitext(video_path)[0] + "_frames.npy"
    preprocess_frames(input_dir, output_file)


Saved processed frames to C:/Users/algba/NikiTechSolutions/Videos/219828_small_frames.npy


In [9]:
# Detecting objects in frames

In [10]:
# Create environment

In [11]:
# conda info --envs

In [12]:
# Create yolov5_env

In [13]:
# conda create -n yolov5_env python=3.9

In [14]:
# Activate New Environment

In [15]:
# conda activate yolov5_env

In [16]:
# Install required packages

In [17]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [18]:
# pip install ultralytics

In [19]:
# pip install notebook ipykernel numpy opencv-python pillow

In [20]:
# Add the environment to jupyter

In [21]:
# python -m ipykernel install --user --name=yolov5_env --display-name "Python (yolov5_env)"

In [22]:
# !jupyter kernelspec list

In [23]:
# import sys
# import torch

# print("Python version:", sys.version)
# print("Torch version:", torch.__version__)
# print("Is CUDA available?", torch.cuda.is_available())

In [24]:
# import torch
# model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
# print("YOLOv5 model loaded successfully!")

In [ ]:
import cv2
from filterpy.kalman import KalmanFilter
import numpy as np
import torch

# Initialize Kalman Filter
kf = KalmanFilter(dim_x=6, dim_z=4)
kf.x = np.zeros(6)  # Initial state (x, y, width, height, vx, vy)
kf.F = np.array([
    [1, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0, 1],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1],
])  # State transition matrix
kf.H = np.array([
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
])  # Measurement matrix
kf.P *= 1000  # Large initial uncertainty
kf.R = np.eye(4) * 10  # Measurement noise
kf.Q = np.eye(6) * 0.1  # Process noise

# Load YOLO model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.hub.load('ultralytics/yolov5', 'yolov5m', pretrained=True).to(device)

# Load video
video_path = "/219828_small.mp4"
cap = cv2.VideoCapture(video_path)

output_path = "/updated_debug_video_with_detections.mp4"
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

frame_number = 0
missed_detections = 0
max_missed_frames = 20  # Allow up to 20 missed frames before resetting

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform YOLO detection
    results = model(frame)
    detections = results.xyxy[0].cpu().numpy()

    # Draw all detected bounding boxes on the frame for debugging
    for det in detections:
        x1, y1, x2, y2, confidence, class_id = det
        if confidence > 0.3:  # Lower confidence threshold
            label = f"Class {int(class_id)}, Conf {confidence:.2f}"
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # Use the largest confident detection (if available) to update Kalman Filter
    if len(detections) > 0:
        detection = max(detections, key=lambda x: x[4])
        x1, y1, x2, y2, confidence, class_id = detection

        if confidence > 0.3 and int(class_id) == 16:  # Bird class ID
            detected_x = (x1 + x2) / 2
            detected_y = (y1 + y2) / 2
            detected_width = x2 - x1
            detected_height = y2 - y1

            # Update Kalman filter
            kf.update([detected_x, detected_y, detected_width, detected_height])
            missed_detections = 0  # Reset missed detection counter
        else:
            missed_detections += 1
            if missed_detections < max_missed_frames:
                kf.predict()
            else:
                kf.x = np.zeros(6)  # Reset Kalman filter
    else:
        missed_detections += 1
        if missed_detections < max_missed_frames:
            kf.predict()
        else:
            kf.x = np.zeros(6)  # Reset Kalman filter

    # Get Kalman Filter's estimated position
    estimated_x = int(kf.x[0])
    estimated_y = int(kf.x[1])
    estimated_width = int(kf.x[2])
    estimated_height = int(kf.x[3])

    # Draw Kalman Filter's bounding box
    if estimated_width > 0 and estimated_height > 0:
        bbox_x1 = estimated_x - estimated_width // 2
        bbox_y1 = estimated_y - estimated_height // 2
        bbox_x2 = estimated_x + estimated_width // 2
        bbox_y2 = estimated_y + estimated_height // 2
        cv2.rectangle(frame, (bbox_x1, bbox_y1), (bbox_x2, bbox_y2), (255, 0, 0), 2)

    # Write frame to output video
    out.write(frame)
    frame_number += 1

cap.release()
out.release()

print(f"Video processing complete. Check the output video at {output_path}")
